# Training YOLO26n dengan Dataset Interval 2

Notebook ini:
- melakukan split stratified per kelas dari `dataset_lele_2_kelas_interval2`
- menjaga grup berdasarkan prefix nama file agar frame dari rekaman yang sama tidak tercampur
- membuat `data.yaml` baru untuk split hasilnya
- training YOLO26n dan export ke NCNN

In [2]:
from pathlib import Path
from collections import defaultdict, Counter
import random
import shutil
import importlib.util
import subprocess
import sys

SOURCE_DIR = Path('dataset_lele_2_kelas_interval2')
SOURCE_IMAGES = SOURCE_DIR / 'train' / 'images'
SOURCE_LABELS = SOURCE_DIR / 'train' / 'labels'
SPLIT_DIR = Path('dataset_split_interval2')
SPLIT_RATIOS = {'train': 0.7, 'valid': 0.2, 'test': 0.1}
SEED = 42
MODEL_PATH = 'yolo26n.pt'
IMG_SIZE = 640
EPOCHS = 100
BATCH = 16
RUNS_DIR = Path('runs/train')
RUN_NAME = 'yolo26_interval2'
CLASS_NAMES = ['Anomali', 'Normal']

def group_key(image_path: Path) -> str:
    parts = image_path.stem.split('_')
    if len(parts) < 3:
        raise ValueError(f'Nama file tidak sesuai pola yang diharapkan: {image_path.name}')
    return '_'.join(parts[:2])

def read_class_id(label_path: Path) -> int:
    text = label_path.read_text(encoding='utf-8').strip()
    if not text:
        raise ValueError(f'Label kosong: {label_path}')
    first_line = next(line for line in text.splitlines() if line.strip())
    return int(float(first_line.split()[0]))

def collect_records() -> list:
    if not SOURCE_IMAGES.exists():
        raise FileNotFoundError(f'Folder gambar sumber tidak ditemukan: {SOURCE_IMAGES}')
    if not SOURCE_LABELS.exists():
        raise FileNotFoundError(f'Folder label sumber tidak ditemukan: {SOURCE_LABELS}')

    records = []
    for image_path in sorted(SOURCE_IMAGES.glob('*.*')):
        label_path = SOURCE_LABELS / f'{image_path.stem}.txt'
        if not label_path.exists():
            raise FileNotFoundError(f'Label tidak ditemukan untuk {image_path.name}')
        class_id = read_class_id(label_path)
        records.append({
            'image_path': image_path,
            'label_path': label_path,
            'class_id': class_id,
            'group': group_key(image_path),
        })
    return records

def allocate_groups(groups: list, ratios: dict, seed: int):
    rng = random.Random(seed)
    groups = groups[:]
    rng.shuffle(groups)
    groups.sort(key=lambda item: (sum(item['class_counts'].values()), len(item['class_counts'])), reverse=True)

    total_class_counts = Counter()
    for group in groups:
        total_class_counts.update(group['class_counts'])

    targets = {
        split_name: Counter({class_id: total_class_counts[class_id] * ratio for class_id in total_class_counts})
        for split_name, ratio in ratios.items()
    }
    current = {split_name: Counter() for split_name in ratios}
    buckets = {split_name: [] for split_name in ratios}

    for group in groups:
        group_total = sum(group['class_counts'].values())

        def score(split_name: str):
            after = 0.0
            before = 0.0
            for class_id in total_class_counts:
                before += abs(current[split_name][class_id] - targets[split_name][class_id])
                after += abs((current[split_name][class_id] + group['class_counts'][class_id]) - targets[split_name][class_id])
            return (after - before, len(buckets[split_name]), abs((sum(current[split_name].values()) + group_total) - sum(targets[split_name].values())))

        split_name = min(ratios.keys(), key=score)
        buckets[split_name].append(group)
        current[split_name].update(group['class_counts'])

    counts = {split_name: sum(current[split_name].values()) for split_name in ratios}
    return buckets, counts, targets

def prepare_split_dirs():
    if SPLIT_DIR.exists():
        shutil.rmtree(SPLIT_DIR)
    for split_name in SPLIT_RATIOS:
        (SPLIT_DIR / split_name / 'images').mkdir(parents=True, exist_ok=True)
        (SPLIT_DIR / split_name / 'labels').mkdir(parents=True, exist_ok=True)

def write_data_yaml():
    yaml_text = '\n'.join([
        f'path: {SPLIT_DIR.as_posix()}',
        'train: train/images',
        'val: valid/images',
        'test: test/images',
        '',
        'nc: 2',
        'names:',
        f'  - {CLASS_NAMES[0]}',
        f'  - {CLASS_NAMES[1]}',
        ''
    ])
    (SPLIT_DIR / 'data.yaml').write_text(yaml_text, encoding='utf-8')

def copy_group(group: list, split_name: str):
    image_out = SPLIT_DIR / split_name / 'images'
    label_out = SPLIT_DIR / split_name / 'labels'
    for record in group:
        shutil.copy2(record['image_path'], image_out / record['image_path'].name)
        shutil.copy2(record['label_path'], label_out / record['label_path'].name)

def build_split_dataset():
    records = collect_records()
    grouped = defaultdict(list)

    for record in records:
        grouped[record['group']].append(record)

    groups = []
    for group_name, group_records in grouped.items():
        class_counts = Counter(record['class_id'] for record in group_records)
        groups.append({
            'name': group_name,
            'records': group_records,
            'class_counts': class_counts,
        })

    prepare_split_dirs()

    split_summary = {name: Counter() for name in SPLIT_RATIOS}
    group_summary = {name: Counter() for name in SPLIT_RATIOS}
    allocated, counts, targets = allocate_groups(groups, SPLIT_RATIOS, SEED)

    for split_name, split_groups in allocated.items():
        for group in split_groups:
            copy_group(group['records'], split_name)
            split_summary[split_name].update(group['class_counts'])
            group_summary[split_name][group['name']] = sum(group['class_counts'].values())

    write_data_yaml()
    return split_summary, group_summary, counts, targets

split_summary, group_summary, split_counts, split_targets = build_split_dataset()
print('Split selesai.')
for split_name in ['train', 'valid', 'test']:
    print(split_name, 'image_count_by_class =', dict(split_summary[split_name]), 'group_count =', split_counts[split_name], 'target =', {k: round(v, 2) for k, v in split_targets[split_name].items()})

Split selesai.
train image_count_by_class = {1: 176, 0: 291} group_count = 467 target = {1: 169.4, 0: 301.7}
valid image_count_by_class = {1: 44, 0: 93} group_count = 137 target = {1: 48.4, 0: 86.2}
test image_count_by_class = {0: 47, 1: 22} group_count = 69 target = {1: 24.2, 0: 43.1}


In [3]:
def count_images_and_labels(split_name: str):
    image_dir = SPLIT_DIR / split_name / 'images'
    label_dir = SPLIT_DIR / split_name / 'labels'
    image_paths = list(image_dir.glob('*.*'))
    label_paths = list(label_dir.glob('*.txt'))
    class_counts = Counter()
    for label_path in label_paths:
        text = label_path.read_text(encoding='utf-8').strip()
        if not text:
            continue
        for line in text.splitlines():
            parts = line.split()
            if len(parts) >= 5:
                class_counts[int(float(parts[0]))] += 1
    return len(image_paths), len(label_paths), dict(class_counts)

for split_name in ['train', 'valid', 'test']:
    image_count, label_count, class_counts = count_images_and_labels(split_name)
    print(split_name, 'images =', image_count, 'labels =', label_count, 'class_counts =', class_counts)

print('data.yaml exists =', (SPLIT_DIR / 'data.yaml').exists())
print('data.yaml path =', SPLIT_DIR / 'data.yaml')

train images = 467 labels = 467 class_counts = {1: 176, 0: 291}
valid images = 137 labels = 137 class_counts = {1: 44, 0: 93}
test images = 69 labels = 69 class_counts = {1: 22, 0: 47}
data.yaml exists = True
data.yaml path = dataset_split_interval2\data.yaml


In [4]:
if importlib.util.find_spec('ultralytics') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'ultralytics'])

from ultralytics import YOLO
import torch

print('torch =', torch.__version__)
print('cuda =', torch.version.cuda)
print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device_name =', torch.cuda.get_device_name(0))

torch = 2.12.0.dev20260408+cu128
cuda = 12.8
cuda_available = True
device_name = NVIDIA GeForce RTX 5060 Laptop GPU


In [5]:
DATA_YAML = SPLIT_DIR / 'data.yaml'
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

model = YOLO(MODEL_PATH)
results = model.train(
    data=str(DATA_YAML),
    imgsz=IMG_SIZE,
    epochs=EPOCHS,
    batch=BATCH,
    device=DEVICE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    seed=SEED
)

results

New https://pypi.org/project/ultralytics/8.4.61 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.10.20 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_split_interval2\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000272FF02F340>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0

In [7]:
BEST_WEIGHTS = Path(r'runs\detect\runs\train\yolo26_interval2\weights\best.pt')
print('best weights =', BEST_WEIGHTS)
if not BEST_WEIGHTS.exists():
    raise FileNotFoundError(f'Bobot terbaik tidak ditemukan: {BEST_WEIGHTS}')

export_model = YOLO(str(BEST_WEIGHTS))
export_results = export_model.export(format='ncnn')
export_results

best weights = runs\detect\runs\train\yolo26_interval2\weights\best.pt
Ultralytics 8.4.51  Python-3.10.20 torch-2.12.0.dev20260408+cu128 CPU (Intel Core i7-14700HX)
WARNING NCNN export does not support end2end models, disabling end2end branch.
YOLO26n summary (fused): 146 layers, 2,495,084 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from 'runs\detect\runs\train\yolo26_interval2\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (5.1 MB)

NCNN: starting export with NCNN 1.0.20260114 and PNNX 20260409...
NCNN: export success  4.4s, saved as 'runs\detect\runs\train\yolo26_interval2\weights\best_ncnn_model' (9.2 MB)

Export complete (4.6s)
Results saved to D:\Kuliah\Skripsi\model-dataset-2-kelas\runs\detect\runs\train\yolo26_interval2\weights\best_ncnn_model
Predict:         yolo predict task=detect model=runs\detect\runs\train\yolo26_interval2\weights\best_ncnn_model imgsz=640 
Validate:        yolo val task=detect model=runs\detect\runs\tr

'runs\\detect\\runs\\train\\yolo26_interval2\\weights\\best_ncnn_model'